# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">9/28(월) 오전 · 예외처리 · 로깅 — 실습</mark>

지난 시간에 만든 함수와 파일 읽기를 오늘 계속 사용합니다. 외우지 말고, 위 예시를 찾아 고쳐 쓰면 됩니다.

오늘 오전의 도착점은 **깨진 줄이 섞인 로그 파일을 넣어도 멈추지 않고 끝까지 처리하는 파서**입니다. 4교시에 `log_parser.py` 로 저장합니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

### 0.1 맨 먼저 · 내 사본 만들기

1. 위 메뉴에서 파일 › 드라이브에 사본 저장을 누릅니다.
2. 제목이 「사본: …」으로 바뀌면 된 것입니다.
3. 사본을 만들지 않으면 내가 쓴 코드가 저장되지 않습니다.

### 0.2 셀 실행하기

셀을 누르고 Shift + Enter를 칩니다. 왼쪽 ▶를 눌러도 같습니다.

### 0.3 오늘 오전의 순서

1. 예외 — 에러가 나면 그 자리에서 멈춘다
2. `try`·`except` — 멈추지 않고 다음 줄로 넘어간다
3. `logging` — 화면 대신 파일에 기록을 남긴다
4. 종합 — 깨진 로그 파일을 끝까지 읽는 파서를 만든다

오늘 필요한 문법만 사용해 한 단계씩 완성합니다.

### 0.4 막혔을 때

1. 문제 아래 ▸ 힌트 보기를 눌러 순서대로 따라 합니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다. 왼쪽 목차(☰)에서 바로 갈 수 있습니다.
3. 정답 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다. 먼저 스스로 해 본 뒤 엽니다.

### 0.5 오늘 만든 코드는 어디에 남나

| | 무엇 |
|---|---|
| 코랩 | 연습장 — 창을 닫으면 여기 쓴 코드는 사라집니다 |
| 내 드라이브의 `agent_core` 폴더 | 작품 보관함 — 날마다 파일이 하나씩 쌓입니다 |

1. 그래서 마지막 실습은 코드를 `.py` 파일로 저장해 드라이브에 남깁니다.
2. 셀 맨 첫 줄의 `%%writefile 이름.py` 는 「이 셀을 실행하지 말고, 이 이름의 파일로 저장하라」는 뜻입니다. 실행 결과 대신 `Writing 이름.py` 가 나옵니다.
3. `!python 이름.py` 는 저장한 파일을 실행합니다. 앞의 `!` 는 「터미널 명령」이라는 표시입니다.
4. 저장된 파일은 drive.google.com 의 `agent_core` 폴더에서 볼 수 있습니다.


In [ ]:
print("9/28 준비 끝")  # 노트북 실행 확인


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">1교시 (09:00–09:50) · 아침 리추얼 30분 + 예외 20분</mark>


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1 · 예외 — 에러가 나면 그 자리에서 멈춘다</mark>

에러를 없애는 시간이 아닙니다. 에러가 **어디서** 나고, 난 뒤에 **무슨 일이 벌어지는지** 눈으로 보는 시간입니다.


### 왜 예외를 알아야 하나

1. 지금까지 만든 코드는 로그가 전부 깔끔하다는 전제 위에 서 있습니다. 칸이 네 개씩 또박또박 들어 있는 파일만 읽어 왔습니다.
2. 실제 로그에는 칸이 모자란 줄, 아예 빈 줄이 섞여 있습니다. 그런 줄을 만나면 파이썬은 그 자리에서 프로그램을 멈춥니다.
3. 20만 줄 가운데 한 줄이 깨졌다고 나머지 19만 9,999줄을 못 읽으면 곤란합니다. 그래서 먼저 **멈추는 장면**을 정확히 알아야 합니다.


### 이 시간에 나오는 말

| 말 | 뜻 | 예 |
|---|---|---|
| 예외 | 실행 도중 생긴 문제로 프로그램이 멈추는 것 | 조리 중 재료가 없어 손이 멈춤 |
| `IndexError` | 리스트에 없는 번호를 꺼낼 때 | `parts[3]` 인데 칸이 2개뿐 |
| `KeyError` | 딕셔너리에 없는 칸 이름을 꺼낼 때 | `log["ip"]` 인데 `ip` 칸이 없음 |
| `ValueError` | 모양이 맞지 않는 값을 넘겼을 때 | `int("abc")` |
| `TypeError` | 갈래가 다른 값끼리 섞었을 때 | `3 + "회"` |
| 트레이스백 | 에러가 난 자리까지의 경로를 보여 주는 빨간 글 | `Traceback (most recent call last)` |


### 에러 메시지를 읽는 규칙 네 가지

1. **맨 아랫줄부터 읽습니다.** 거기에 에러 이름과 한 줄 설명이 있습니다.
2. 에러 **이름**을 먼저 봅니다. `IndexError` 인지 `KeyError` 인지에 따라 고칠 곳이 다릅니다.
3. 그 위의 `line …` 은 **몇 번째 줄에서 멈췄는지**입니다. 그 줄만 보면 됩니다.
4. 빨간 글이 길다고 어려운 것이 아닙니다. 쓸모 있는 정보는 대개 두 줄뿐입니다.


<div style="background:#e8f5e9; color:#17351f; padding:12px 16px; border-radius:8px"><strong>🔎 이 절의 셀은 일부러 에러가 납니다</strong><br>빨간 글이 나오면 잘못한 것이 아니라 성공한 것입니다. 에러 이름을 소리 내어 읽고 다음 셀로 넘어가세요.</div>


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.1 리스트에 없는 번호를 꺼냈을 때</mark>

칸이 모자란 줄을 쪼개면 리스트가 짧아집니다. 없는 번호를 꺼내면 `IndexError` 가 납니다.


In [ ]:
line = "03:22,hacker"        # 칸이 두 개뿐인 깨진 줄
parts = line.strip().split(",")
print(parts)                 # ['03:22', 'hacker'] — 0번과 1번뿐입니다
print(parts[2])              # 없는 2번을 꺼내면 여기서 멈춥니다


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.2 딕셔너리에 없는 칸 이름을 꺼냈을 때</mark>

리스트는 번호가 없을 때, 딕셔너리는 **이름**이 없을 때 멈춥니다. 에러 이름이 `KeyError` 로 달라집니다.


In [ ]:
log = {"time": "03:22", "user": "hacker"}   # ip 칸이 없는 로그 한 건
print(log["user"])                          # 있는 칸은 잘 나옵니다
print(log["ip"])                            # 없는 칸을 꺼내면 여기서 멈춥니다


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.3 멈춘 뒤의 줄은 실행되지 않는다</mark>

이것이 오늘의 핵심 장면입니다. 반복문 한가운데서 에러가 나면 **남은 줄도, 남은 반복도** 전부 사라집니다.


In [ ]:
lines = ["09:01,kim01,LOGIN,10.0.3.21", "03:22,hacker", "09:05,lee02,LOGIN,10.0.7.5"]

for line in lines:
    parts = line.strip().split(",")
    print("읽는 중:", parts[1], parts[3])   # 두 번째 줄에서 멈춥니다

print("끝까지 읽었습니다")                   # 이 줄은 영영 실행되지 않습니다


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-1 · 에러 이름 확인하기</font></h3></td></tr></table>

주어진 리스트에서 **3번 칸**을 꺼내 출력하시오. 에러가 나는 것이 정답입니다.

1. `parts` 를 `print()` 로 먼저 출력해 칸이 몇 개인지 눈으로 봅니다.
2. 그다음 줄에서 `parts[3]` 을 출력합니다.
3. 빨간 글 **맨 아랫줄**에 나온 에러 이름을 읽습니다.

| | |
|---|---|
| 주어지는 값 | `parts = ["09:01", "kim01"]` |
| 🎯 나와야 하는 결과 | `IndexError: list index out of range` |

<details>
<summary>▸ 힌트 보기</summary>

1. 리스트의 값을 꺼낼 때는 대괄호에 번호를 씁니다: `parts[0]`.
2. 번호는 0부터 셉니다. 값이 두 개면 있는 번호는 0과 1뿐입니다.
3. `print()` 안에 `parts[3]` 을 그대로 넣습니다.
4. 빨간 글이 나오면 성공입니다. 맨 아랫줄만 읽으면 됩니다.

그래도 막히면 맨 아래 정답 1-1 셀의 「코드 표시」를 누릅니다.

</details>


In [ ]:
parts = ["09:01", "kim01"]


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-2 · 없는 칸 이름 꺼내기</font></h3></td></tr></table>

주어진 로그에서 `result` 칸을 꺼내 출력하시오. 이 로그에는 `result` 칸이 없습니다.

1. 있는 칸인 `user` 를 먼저 꺼내 출력해 코드가 맞는지 확인합니다.
2. 그다음 줄에서 `result` 칸을 꺼내 출력합니다.
3. 에러 이름이 문제 1-1과 **다르다**는 것을 확인합니다.

| | |
|---|---|
| 주어지는 값 | `log = {"time": "09:01", "user": "kim01"}` |
| 🎯 나와야 하는 결과 | `KeyError: 'result'` |

<details>
<summary>▸ 힌트 보기</summary>

1. 딕셔너리에서 값을 꺼낼 때는 대괄호에 **칸 이름**을 따옴표까지 써 넣습니다: `log["user"]`.
2. 번호가 아니라 이름입니다. `log[1]` 이 아닙니다.
3. `print(log["result"])` 한 줄이면 됩니다.
4. 에러 이름이 `KeyError` 로 나오면 성공입니다.

그래도 막히면 맨 아래 정답 1-2 셀의 「코드 표시」를 누릅니다.

</details>


In [ ]:
log = {"time": "09:01", "user": "kim01"}


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 1-3 · 몇 건까지 읽고 멈추나</font></h3></td></tr></table>

깨진 줄이 섞인 목록을 반복문으로 읽으면서, 읽은 줄의 **계정 이름**을 한 줄씩 출력하시오. 도중에 멈추는 것이 정답입니다.

1. 빈 줄이 아니라 **칸이 모자란 줄**이 섞여 있습니다.
2. `for` 로 `lines` 를 하나씩 반복합니다.
3. 반복 안에서 줄을 쉼표로 쪼개고, **1번 칸**과 **3번 칸**을 함께 출력합니다.
4. 반복문 바깥 마지막 줄에 `print("끝")` 을 두고, 그 줄이 실행되는지 봅니다.

| | |
|---|---|
| 주어지는 값 | 아래 셀의 `lines` (3줄 · 가운데가 깨진 줄) |
| 🎯 나와야 하는 결과 | 첫 줄만 출력된 뒤 `IndexError` · `끝` 은 출력되지 않음 |

<details>
<summary>▸ 힌트 보기</summary>

1. 줄을 쪼개는 법은 지난 시간에 배운 `line.strip().split(",")` 입니다.
2. 쪼갠 결과를 담을 이름을 하나 정합니다(예: `parts`).
3. 한 줄에 두 값을 출력할 때는 `print(a, b)` 처럼 쉼표로 나열합니다.
4. 마지막 `print("끝")` 은 **들여쓰지 않습니다.** 들여쓰면 반복 안으로 들어갑니다.

그래도 막히면 맨 아래 정답 1-3 셀의 「코드 표시」를 누릅니다.

</details>


In [ ]:
lines = [
    "09:01,kim01,LOGIN,10.0.3.21",
    "03:22,hacker",
    "09:05,lee02,LOGIN,10.0.7.5",
]


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>위 세 문제를 다 푼 사람만 풉니다. 처음 코딩하는 분은 건너뛰고 다음 절로 가세요. 새 문법은 나오지 않습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 1-1 · 없는 파일을 열면</font></h3></td></tr></table>

아직 만들지 않은 파일을 읽으려고 하시오. 어떤 에러 이름이 나오는지 확인하는 문제입니다.

1. 지난 시간에 배운 `with open(...) as f:` 로 `no_such_file.csv` 를 읽기 모드로 엽니다.
2. 연 파일의 내용을 `f.read()` 로 읽어 출력합니다.

| | |
|---|---|
| 주어지는 값 | 파일 이름 `no_such_file.csv` |
| 🎯 나와야 하는 결과 | `FileNotFoundError` 로 시작하는 에러 |

<details>
<summary>▸ 힌트 보기</summary>

1. 읽기 모드는 `open("이름", encoding="utf-8")` 입니다. 모드를 적지 않으면 읽기입니다.
2. `with` 줄 끝에는 콜론을 붙이고, 파일을 쓰는 줄은 네 칸 들여씁니다.
3. 이 에러는 9/23에 한 번 본 적이 있습니다.

</details>


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 1-2 · 숫자와 글자를 섞으면</font></h3></td></tr></table>

실패 횟수 뒤에 「회」를 붙여 한 덩어리로 만들려다 나는 에러를 확인하시오.

1. `fail_count` 에 숫자 `3` 을 담습니다.
2. `fail_count + "회"` 를 출력해 봅니다.
3. 에러를 확인한 뒤, **아래 줄에** f-string(`f"{fail_count}회"`)으로 제대로 출력하는 줄을 덧붙여 비교합니다. 단, 에러 나는 줄을 지우지는 않습니다.

| | |
|---|---|
| 주어지는 값 | `fail_count = 3` |
| 🎯 나와야 하는 결과 | `TypeError` 로 시작하는 에러 |

<details>
<summary>▸ 힌트 보기</summary>

1. 숫자와 글자는 `+` 로 이어 붙일 수 없습니다. 파이썬이 어느 쪽으로 맞춰야 할지 모릅니다.
2. f-string 은 따옴표 앞에 `f` 를 붙이고 중괄호 안에 이름을 넣는 방식입니다.
3. 에러가 난 줄에서 멈추므로, 뒤에 덧붙인 f-string 줄은 실행되지 않습니다. 그것까지 확인하면 이 절을 다 이해한 것입니다.

</details>


In [ ]:
fail_count = 3


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 1-3 · 함수 안에서 에러가 나면</font></h3></td></tr></table>

지난 시간에 만든 것과 같은 모양의 함수를 만들고, 깨진 줄을 넘겨 보시오. **부른 쪽**도 함께 멈춘다는 것을 확인하는 문제입니다.

1. 줄 하나를 받아 딕셔너리로 돌려주는 함수를 만듭니다. 함수 이름은 `parse_line` 으로 정합니다.
2. 함수 안에서 줄을 쉼표로 쪼개고, `time`·`user`·`action`·`ip` 네 칸을 담은 딕셔너리를 `return` 합니다.
3. 정상 줄로 한 번 불러 결과를 출력합니다.
4. 깨진 줄로 한 번 더 부릅니다.
5. 맨 마지막에 `print("함수 호출을 마쳤습니다")` 를 두고, 그 줄이 실행되는지 봅니다.

| | |
|---|---|
| 주어지는 값 | 정상 줄 `"09:01,kim01,LOGIN,10.0.3.21"` · 깨진 줄 `"03:22,hacker"` |
| 🎯 나와야 하는 결과 | 첫 호출의 딕셔너리가 출력된 뒤 `IndexError` · 마지막 줄은 출력되지 않음 |

<details>
<summary>▸ 힌트 보기</summary>

1. 함수는 `def 이름(매개변수):` 로 시작하고, 안에서 할 일은 네 칸 들여씁니다.
2. 딕셔너리를 만들 때 칸 이름과 값은 콜론으로 잇습니다: `{"time": parts[0], ...}`.
3. 에러가 난 자리는 함수 **안**이지만, 멈추는 것은 프로그램 **전체**입니다.

</details>


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.정리 📋 한눈에</mark>

| 에러 이름 | 언제 나나 |
|---|---|
| `IndexError` | 리스트에 없는 **번호**를 꺼냈다 |
| `KeyError` | 딕셔너리에 없는 **칸 이름**을 꺼냈다 |
| `ValueError` | 모양이 맞지 않는 값을 넘겼다 (`int("abc")`) |
| `TypeError` | 갈래가 다른 값을 섞었다 (`3 + "회"`) |
| `FileNotFoundError` | 없는 파일을 열려고 했다 |

기억할 것은 하나입니다. **에러가 난 줄에서 프로그램 전체가 멈춥니다.** 다음 시간에 멈추지 않는 법을 배웁니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

각 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다. 정답 셀은 혼자서 실행됩니다.


In [ ]:
#@title 정답 1-1 { display-mode: "form" }
parts = ["09:01", "kim01"]

print(parts)
print(parts[3])


In [ ]:
#@title 정답 1-2 { display-mode: "form" }
log = {"time": "09:01", "user": "kim01"}

print(log["user"])
print(log["result"])


In [ ]:
#@title 정답 1-3 { display-mode: "form" }
lines = [
    "09:01,kim01,LOGIN,10.0.3.21",
    "03:22,hacker",
    "09:05,lee02,LOGIN,10.0.7.5",
]

for line in lines:
    parts = line.strip().split(",")
    print(parts[1], parts[3])

print("끝")


In [ ]:
#@title 정답 ⭐1-1 { display-mode: "form" }
with open("no_such_file.csv", encoding="utf-8") as f:
    print(f.read())


In [ ]:
#@title 정답 ⭐1-2 { display-mode: "form" }
fail_count = 3

print(fail_count + "회")     # TypeError 로 여기서 멈춥니다
print(f"{fail_count}회")     # 위에서 멈추므로 이 줄은 실행되지 않습니다


In [ ]:
#@title 정답 ⭐1-3 { display-mode: "form" }
def parse_line(line):
    parts = line.strip().split(",")
    return {"time": parts[0], "user": parts[1], "action": parts[2], "ip": parts[3]}

print(parse_line("09:01,kim01,LOGIN,10.0.3.21"))
print(parse_line("03:22,hacker"))          # 함수 안에서 IndexError 가 납니다

print("함수 호출을 마쳤습니다")               # 이 줄은 실행되지 않습니다
